# A Perfect Score for Saying Nothing

*We scored a visa assistant's polite refusals with DeepEval. Faithfulness passed every one of them.*

**Read this if**

- **You build the eval suite.** You leave with a paired test that shows whether a metric sees your worst failure, and why the obvious Faithfulness fix fails.
- **You know what a good visa answer looks like.** You leave knowing which part of your judgement these scores cannot make, put in the words you already use.
- **You sign off on the number.** You leave knowing what a high relevancy and faithfulness score lets the team claim, and what it does not.

An eval is an automated check that scores what a model writes. Teams that ship an assistant which looks documents up before answering usually start with two of them. One asks whether the answer stays on the question. The other asks whether it sticks to the documents. Between them they cover the failures everyone worries about, rambling and made-up facts. So it feels safe to believe that if faithfulness and relevancy are both high, the answer is good.

Fernway is a travel booking platform with a visa assistant built into its booking flow. A traveller enters a passport country and a destination, and the assistant lists the documents they need. Travellers act on that list directly. Nobody at Fernway reads it first.

When the assistant isn't sure, it writes a polite deflection instead. A traveller who gets one on a booking site books somewhere else. Then Fernway's support team takes the call. Here is the same question, answered both ways by the assistant we build in this notebook.

> **Traveller:** I have a passport from India. What documents do I need to travel to Japan?
>
> **Good answer:** To travel to Japan with an Indian passport, you need a tourist visa issued before travel. For the visa application, you must provide:
> - A passport valid for at least six months
> - A completed visa application form
> - One recent passport photo
> - A flight itinerary
> - Bank statements for the last three months
>
> **Deflection:** To ensure you have the most accurate and up-to-date information for your trip to Japan, we kindly recommend checking the official website of the Embassy or Consulate General of Japan. They will provide the exact document and visa requirements based on your specific travel details.

Both replies stay on the question, and neither contradicts Fernway's policy. Only the first one helps the traveller. We wanted to know whether DeepEval's AnswerRelevancy and Faithfulness metrics could tell them apart.

## The assistant that answers nothing

We build the assistant with LangGraph, a library that wires model calls into a small graph of steps. A lookup step finds the policy passage for the destination. A router then hands the request to one of two writers. The first answers from the passage. The second writes the deflection Fernway's assistant produces when it is unsure.

The assistant and the judge both run on `gemini-3.8-flash`. The judge is a second model call that reads an answer and rules on it. Running the notebook needs a `GEMINI_API_KEY` in the environment.

In [1]:
import asyncio
import os

# deepeval reads this when it is imported, so it is set first
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

from importlib.metadata import version
from typing import TypedDict

from deepeval.errors import MissingTestCaseParamsError
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric, GEval
from deepeval.models import GeminiModel
from deepeval.test_case import LLMTestCase, SingleTurnParams
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph

if "GEMINI_API_KEY" not in os.environ:
    raise RuntimeError("Set the GEMINI_API_KEY environment variable before running this notebook.")

MODEL = "gemini-3.8-flash"
THRESHOLD = 0.7  # minimum passing score for every metric below
RUNS = 3

assistant_llm = ChatGoogleGenerativeAI(model=MODEL, google_api_key=os.environ["GEMINI_API_KEY"])

print(f"deepeval               {version('deepeval')}")
print(f"langgraph              {version('langgraph')}")
print(f"langchain-google-genai {version('langchain-google-genai')}")
print(f"judge and assistant    {MODEL}")

deepeval               4.3.1
langgraph              1.2.20
langchain-google-genai 4.4.3
judge and assistant    gemini-3.8-flash


The setup cell pins deepeval 4.3.1, langgraph 1.2.20 and langchain-google-genai 4.4.3, the versions behind every number in this article.

The data is small on purpose: eight travellers, each a passport country and a destination. The policy is six short passages, one per destination. We wrote those rules for this experiment, so they are not real or current visa rules.

In [2]:
# Invented for this experiment. These are not real or current visa rules.
POLICY = {
    "Japan": (
        "Japan. Holders of United States passports do not need a visa for tourist stays of up to 90 days. "
        "They need a passport valid for the whole stay and a return or onward ticket. "
        "Holders of Indian passports need a tourist visa issued before travel. "
        "The visa application needs a passport valid for at least six months, a completed visa application form, "
        "one recent passport photo, a flight itinerary, and bank statements for the last three months. "
        "Processing takes five working days."
    ),
    "Thailand": (
        "Thailand. Holders of Indian passports can get a visa on arrival for tourist stays of up to 15 days. "
        "At the airport they must show a passport valid for at least six months, one passport photo, "
        "a confirmed return ticket within 15 days, proof of accommodation, and the visa fee of 2,000 baht in cash. "
        "Holders of United States passports can enter without a visa for up to 60 days."
    ),
    "the United Kingdom": (
        "United Kingdom. Holders of United States passports need an Electronic Travel Authorisation before travel "
        "for visits of up to six months. It is applied for online with the passport, a photo of the traveller's face, "
        "and a payment card. Holders of Indian passports need a Standard Visitor visa before travel. "
        "The application needs a passport, bank statements for the last six months, a letter from the traveller's "
        "employer, and an appointment at a visa centre to give fingerprints and a photo."
    ),
    "France": (
        "France (Schengen area). Holders of Indian passports need a Schengen short-stay visa before travel. "
        "The application needs a passport issued in the last ten years and valid for three months after the return date, "
        "a completed application form, two passport photos, travel medical insurance covering at least 30,000 euros, "
        "a return flight reservation, hotel bookings for every night, and bank statements for the last three months."
    ),
    "Canada": (
        "Canada. Holders of United Kingdom passports flying to Canada need an Electronic Travel Authorization "
        "before boarding. It is applied for online with the passport, an email address, and a payment card, "
        "and costs 7 Canadian dollars. Holders of Indian passports need a visitor visa and must give "
        "fingerprints at a visa application centre."
    ),
    "Singapore": (
        "Singapore. Holders of Indian passports need a visa before travel, applied for through an authorised visa agent. "
        "The application needs a passport valid for at least six months, a completed Form 14A, one recent passport photo, "
        "and a return ticket. Every traveller, whatever their passport, must also submit the SG Arrival Card online "
        "within three days before arrival."
    ),
}

CASES = [
    {"id": "c1", "passport": "India", "destination": "Japan"},
    {"id": "c2", "passport": "the United States", "destination": "Japan"},
    {"id": "c3", "passport": "India", "destination": "Thailand"},
    {"id": "c4", "passport": "India", "destination": "the United Kingdom"},
    {"id": "c5", "passport": "the United States", "destination": "the United Kingdom"},
    {"id": "c6", "passport": "India", "destination": "France"},
    {"id": "c7", "passport": "the United Kingdom", "destination": "Canada"},
    {"id": "c8", "passport": "India", "destination": "Singapore"},
]

for case in CASES:
    print(f"{case['id']}  passport from {case['passport']:<18} to {case['destination']}")

c1  passport from India              to Japan
c2  passport from the United States  to Japan
c3  passport from India              to Thailand
c4  passport from India              to the United Kingdom
c5  passport from the United States  to the United Kingdom
c6  passport from India              to France
c7  passport from the United Kingdom to Canada
c8  passport from India              to Singapore


Each passage covers more than one passport, so a good answer has to pick the right row. Answering each traveller twice gives us sixteen test cases. A test case is one scored example. Ours holds the traveller's question, the assistant's reply and the policy passage the assistant looked up. DeepEval calls those fields `input`, `actual_output` and `retrieval_context`. The last name means the text the assistant retrieved and was supposed to rely on.

In [3]:
class VisaState(TypedDict):
    question: str
    destination: str
    unsure: bool
    policy: str
    answer: str


ANSWER_PROMPT = """You are Fernway's visa assistant on a travel booking site.
Answer the traveller's question using only the policy text below.
Name every document they need. Keep the answer under 80 words.

Policy text:
{policy}

Traveller: {question}"""

DEFLECT_PROMPT = """You are Fernway's visa assistant on a travel booking site.
You are not confident about this traveller's requirements, so do not state any.
Write a polite reply of one or two sentences that points them to the official embassy or consulate website.

Traveller: {question}"""


def look_up_policy(state):
    return {"policy": POLICY[state["destination"]]}


def write_answer(state):
    reply = assistant_llm.invoke(ANSWER_PROMPT.format(policy=state["policy"], question=state["question"]))
    return {"answer": reply.text}


def write_deflection(state):
    reply = assistant_llm.invoke(DEFLECT_PROMPT.format(question=state["question"]))
    return {"answer": reply.text}


def route(state):
    if state["unsure"]:
        return "write_deflection"
    return "write_answer"


graph = StateGraph(VisaState)
graph.add_node("look_up_policy", look_up_policy)
graph.add_node("write_answer", write_answer)
graph.add_node("write_deflection", write_deflection)
graph.add_edge(START, "look_up_policy")
graph.add_conditional_edges("look_up_policy", route)
graph.add_edge("write_answer", END)
graph.add_edge("write_deflection", END)
assistant = graph.compile()

for node_name in assistant.get_graph().nodes:
    print(node_name)

__start__
look_up_policy
write_answer
write_deflection
__end__


The printed nodes are the lookup, the two writers, and LangGraph's own start and end markers.

We force the unsure branch on one of each traveller's two runs. In production a confidence check would make that call. Forcing it is a shortcut. It also makes the metrics' job easier than it would be in real use. Every deflection here dodges a question the policy answers in full. A real unsure branch fires on harder questions, where deflecting is sometimes the honest reply.

In [4]:
TEST_CASES = {}  # TEST_CASES[case id] -> LLMTestCase, e.g. "c1-good", "c1-deflect"

for case in CASES:
    question = f"I have a passport from {case['passport']}. What documents do I need to travel to {case['destination']}?"
    for kind in ["good", "deflect"]:
        unsure = kind == "deflect"
        state = assistant.invoke({"question": question, "destination": case["destination"], "unsure": unsure})
        test_case = LLMTestCase(input=question, actual_output=state["answer"], retrieval_context=[state["policy"]])
        TEST_CASES[f"{case['id']}-{kind}"] = test_case

for case_id, test_case in TEST_CASES.items():
    print(f"[{case_id}] {test_case.input}")
    print(f"    {test_case.actual_output}")
    print()

[c1-good] I have a passport from India. What documents do I need to travel to Japan?
    To travel to Japan with an Indian passport, you need a tourist visa issued before travel. 

For the visa application, you must provide:
- A passport valid for at least six months
- A completed visa application form
- One recent passport photo
- A flight itinerary
- Bank statements for the last three months

[c1-deflect] I have a passport from India. What documents do I need to travel to Japan?
    To ensure you have the most accurate and up-to-date information for your trip to Japan, we kindly recommend checking the official website of the Embassy or Consulate General of Japan. They will provide the exact document and visa requirements based on your specific travel details.

[c2-good] I have a passport from the United States. What documents do I need to travel to Japan?
    For tourist stays of up to 90 days in Japan, you do not need a visa. You will need the following documents:

* A passport vali

The pattern is easy to see by eye. Every good answer lists documents from its passage, and every deflection lists none. All eight deflections point the traveller to an embassy or government website. Three of them, c3, c7 and c8, also wish the traveller safe travels or thank them for using Fernway.

## What relevancy and faithfulness each compare

[AnswerRelevancy](https://deepeval.com/docs/metrics-answer-relevancy) asks whether the reply stays on the question. It splits the reply into statements, and the judge gives each one a verdict. A verdict is a one-word ruling: yes, no or borderline. DeepEval's shipped prompt describes borderline as ambiguous or supporting. The score is the share of statements that pass.

[Faithfulness](https://deepeval.com/docs/metrics-faithfulness) asks whether the reply agrees with the policy passage. It pulls a list of facts out of the passage and a list of claims out of the reply. The judge then rules on each claim against the facts. The score is again the share that pass.

Neither metric asks what the traveller cares about, which is whether the reply names the documents. We wanted to see whether that gap shows up in the scores.

To watch the metrics work, we give each measurement its own judge that records every prompt it receives. That record lets us count judge calls exactly. It also tells us which test case fields actually reach the judge.

In [5]:
def recording_judge(prompts):
    """Return a fresh judge that appends every prompt it is sent to the given list."""
    # GeminiModel falls back to GOOGLE_API_KEY unless the key is passed in
    judge = GeminiModel(model=MODEL, api_key=os.environ["GEMINI_API_KEY"])
    plain_a_generate = judge.a_generate

    async def recording_a_generate(prompt, *args, **kwargs):
        # one entry per judge call; retries inside the model client are not counted
        prompts.append(prompt)
        return await plain_a_generate(prompt, *args, **kwargs)

    judge.a_generate = recording_a_generate
    return judge


METRIC_CLASSES = {
    "AnswerRelevancy": AnswerRelevancyMetric,
    "Faithfulness": FaithfulnessMetric,
}
EXTRA_ARGS = {}  # metric name -> keyword arguments beyond model and threshold


def build_metric(name, judge):
    """Return a fresh metric instance, so no state carries over between test cases."""
    extra = EXTRA_ARGS.get(name, {})
    return METRIC_CLASSES[name](model=judge, threshold=THRESHOLD, **extra)


async def measure_once(name, test_case):
    """Score one test case with its own judge and return the score, the metric and the prompts sent."""
    prompts = []
    metric = build_metric(name, recording_judge(prompts))
    # the progress bar is a live widget, and live widgets break headless notebook runs
    await metric.a_measure(test_case, _show_indicator=False)
    return {"score": metric.score, "calls": len(prompts), "prompts": prompts, "metric": metric}


for name, metric_class in METRIC_CLASSES.items():
    declared = ""
    for param in metric_class._required_params:
        declared += f" {param.value}"
    print(f"{name:<16} declares it needs:{declared}")

AnswerRelevancy  declares it needs: input actual_output
Faithfulness     declares it needs: input actual_output retrieval_context


By their own declarations, Faithfulness needs the question, the reply and the passage, while AnswerRelevancy needs only the question and the reply. We put both declarations to the test further down. First comes one pair in full: the India to Japan question, answered well and then deflected.

In [6]:
for name in METRIC_CLASSES:
    for case_id in ["c1-good", "c1-deflect"]:
        result = await measure_once(name, TEST_CASES[case_id])
        metric = result["metric"]
        print(f"{name} on {case_id}")
        for verdict in metric.verdicts:
            print(f"    verdict: {verdict.verdict.value}")
        print(f"    score:   {result['score']:.2f}")
        print(f"    reason:  {metric.reason}")
        print()

AnswerRelevancy on c1-good
    verdict: yes
    verdict: yes
    verdict: yes
    verdict: yes
    verdict: yes
    verdict: yes
    score:   1.00
    reason:  The score is 1.00 because the response directly and effectively addresses the required travel documents for an Indian passport holder visiting Japan, with zero irrelevant information included. Excellent work!



AnswerRelevancy on c1-deflect
    verdict: borderline
    verdict: borderline
    score:   1.00
    reason:  The score is 1.00 because the response perfectly and directly addresses all the necessary travel documents for an Indian passport holder traveling to Japan, with no irrelevant details included. Excellent work!



Faithfulness on c1-good
    verdict: yes
    verdict: yes
    verdict: yes
    verdict: yes
    verdict: yes
    verdict: yes
    score:   1.00
    reason:  The score is 1.00 because there are no contradictions, and the output perfectly aligns with the provided context.



Faithfulness on c1-deflect
    verdict: borderline
    verdict: borderline
    score:   1.00
    reason:  The score is 1.00 because the output is completely faithful to the context with no contradictions at all, fantastic job!



The good answer gets a yes on every statement from both metrics and scores 1.00 on each.

The deflection is where things stop making sense. AnswerRelevancy marked both of its statements borderline and still gave it 1.00. So the judge never called the deflection relevant. It hedged on every sentence, and DeepEval counted each hedge as a pass.

The written reason makes it worse. It praises the deflection for addressing the required documents, though the reply names none of them. Faithfulness does the same. Both its claims get a borderline verdict, the score is 1.00, and the reason celebrates zero contradictions. The judge had its doubts, and the scoring step threw them away.

## Sixteen answers, two metrics, three runs

One pair is a lead, not a finding, so we score all sixteen cases with both metrics, three times over.

The bar is separation. A metric separates when every deflection scores below the good answer to the same question. That has to hold on every run. The judge's rulings can shift from one call to the next, which is why a single clean run proves little.

A threshold is the minimum score that counts as a pass. We set ours once in the setup cell, and every report below prints it.

In [7]:
SCORES = {}    # SCORES[metric][run][case id] -> score
CALLS = {}     # CALLS[metric][run][case id] -> judge calls for that one measurement
PROMPTS = {}   # PROMPTS[metric][run][case id] -> list of prompts the judge received
TALLIES = {}   # TALLIES[metric][run][case id] -> {"yes": n, "borderline": n, "no": n}


async def run_suite(names):
    """Measure every test case with every named metric, RUNS times, and fill the four tables."""
    for name in names:
        SCORES[name] = {}
        CALLS[name] = {}
        PROMPTS[name] = {}
        TALLIES[name] = {}
        for run in range(1, RUNS + 1):
            SCORES[name][run] = {}
            CALLS[name][run] = {}
            PROMPTS[name][run] = {}
            TALLIES[name][run] = {}
            pending = []
            for test_case in TEST_CASES.values():
                pending.append(measure_once(name, test_case))
            # this loop is about volume: the cases in one run are independent, so they are scored
            # concurrently, and each has its own recording judge, so call counts stay separate
            results = await asyncio.gather(*pending)
            for case_id, result in zip(TEST_CASES, results):
                tally = {"yes": 0, "borderline": 0, "no": 0}
                for verdict in getattr(result["metric"], "verdicts", []):
                    tally[verdict.verdict.value] += 1
                SCORES[name][run][case_id] = result["score"]
                CALLS[name][run][case_id] = result["calls"]
                PROMPTS[name][run][case_id] = result["prompts"]
                TALLIES[name][run][case_id] = tally


def print_scores(name):
    print(f"{name}: one score per run")
    for case_id in TEST_CASES:
        line = f"    {case_id:<11}"
        for run in range(1, RUNS + 1):
            line += f"  {SCORES[name][run][case_id]:.2f}"
        print(line)


FIRST_PAIR = ["AnswerRelevancy", "Faithfulness"]
await run_suite(FIRST_PAIR)
for name in FIRST_PAIR:
    print_scores(name)
    print()

AnswerRelevancy: one score per run
    c1-good      1.00  1.00  1.00
    c1-deflect   1.00  1.00  1.00
    c2-good      1.00  1.00  1.00
    c2-deflect   1.00  1.00  1.00
    c3-good      1.00  1.00  1.00
    c3-deflect   0.50  0.50  0.50
    c4-good      1.00  1.00  1.00
    c4-deflect   1.00  1.00  1.00
    c5-good      1.00  1.00  1.00
    c5-deflect   1.00  1.00  1.00
    c6-good      1.00  1.00  1.00
    c6-deflect   1.00  1.00  1.00
    c7-good      1.00  1.00  1.00
    c7-deflect   0.50  0.50  0.50
    c8-good      1.00  1.00  1.00
    c8-deflect   0.75  0.67  0.50

Faithfulness: one score per run
    c1-good      1.00  1.00  1.00
    c1-deflect   1.00  1.00  1.00
    c2-good      1.00  0.67  1.00
    c2-deflect   1.00  1.00  1.00
    c3-good      1.00  1.00  1.00
    c3-deflect   1.00  1.00  1.00
    c4-good      1.00  1.00  1.00
    c4-deflect   1.00  1.00  1.00
    c5-good      1.00  1.00  1.00
    c5-deflect   1.00  1.00  1.00
    c6-good      1.00  1.00  1.00
    c6-deflect

Faithfulness gave every deflection 1.00 on every run.

AnswerRelevancy gave five of the eight deflections 1.00 on every run. The other three were c3 and c7, at 0.50 on all three runs, and c8, which slid from 0.75 to 0.67 to 0.50. Those are exactly the three deflections with a sign-off attached. Refusing to answer cost nothing, since all eight refuse.

Faithfulness also marked down two good answers now and then. Case c2-good dropped to 0.67 on one run, and c7-good to 0.75 on two.

In [8]:
def separates(name, run):
    """True when every deflection scores strictly below the good answer to the same question."""
    for case in CASES:
        good_score = SCORES[name][run][case["id"] + "-good"]
        deflection_score = SCORES[name][run][case["id"] + "-deflect"]
        if deflection_score >= good_score:
            return False
    return True


def report_separation(name):
    runs_separated = 0
    good_total = 0.0
    deflection_total = 0.0
    good_passing = 0
    deflections_passing = 0
    measurements = 0
    for run in range(1, RUNS + 1):
        if separates(name, run):
            runs_separated += 1
        for case in CASES:
            good_score = SCORES[name][run][case["id"] + "-good"]
            good_total += good_score
            if good_score >= THRESHOLD:
                good_passing += 1
            deflection_score = SCORES[name][run][case["id"] + "-deflect"]
            deflection_total += deflection_score
            if deflection_score >= THRESHOLD:
                deflections_passing += 1
            measurements += 1
    good_mean = good_total / measurements
    deflection_mean = deflection_total / measurements
    print(name)
    print(f"    runs where every deflection scored lower: {runs_separated} of {RUNS}")
    print(f"    mean score, good answers:                 {good_mean:.2f}")
    print(f"    mean score, deflections:                  {deflection_mean:.2f}")
    print(f"    good answers passing at {THRESHOLD}:            {good_passing} of {measurements}")
    print(f"    deflections passing at {THRESHOLD}:             {deflections_passing} of {measurements}")


for name in FIRST_PAIR:
    report_separation(name)

AnswerRelevancy
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 1.00
    mean score, deflections:                  0.83
    good answers passing at 0.7:            24 of 24
    deflections passing at 0.7:             16 of 24
Faithfulness
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 0.97
    mean score, deflections:                  1.00
    good answers passing at 0.7:            23 of 24
    deflections passing at 0.7:             24 of 24


Neither metric separated on a single run.

AnswerRelevancy passed 16 of 24 deflections at the 0.7 threshold, with a mean of 0.83, and passed all 24 good answers. Faithfulness did worse. It passed 24 of 24 deflections but only 23 of 24 good answers. Its mean for deflections, 1.00, ended up above its mean for good answers, 0.97.

Anyone watching a dashboard built on these two numbers would see a healthy assistant.

## Why a refusal is the cheapest perfect score there is

Our starting hypothesis came with a reason. Deflections would look relevant to the judge because they are on topic. They would look faithful because they make almost no claims. The judge's own verdicts, summed over every run, let us check that reason directly.

In [9]:
def print_tallies(name):
    for kind in ["good", "deflect"]:
        totals = {"yes": 0, "borderline": 0, "no": 0}
        empty_cases = 0
        for run in range(1, RUNS + 1):
            for case in CASES:
                tally = TALLIES[name][run][case["id"] + "-" + kind]
                items = tally["yes"] + tally["borderline"] + tally["no"]
                if items == 0:
                    empty_cases += 1
                for verdict_value in totals:
                    totals[verdict_value] += tally[verdict_value]
        print(f"{name}, {kind}: yes {totals['yes']}, borderline {totals['borderline']}, no {totals['no']}, "
              f"measurements with nothing to judge {empty_cases}")


for name in FIRST_PAIR:
    print_tallies(name)

AnswerRelevancy, good: yes 104, borderline 32, no 0, measurements with nothing to judge 0
AnswerRelevancy, deflect: yes 2, borderline 55, no 13, measurements with nothing to judge 0
Faithfulness, good: yes 93, borderline 17, no 3, measurements with nothing to judge 0
Faithfulness, deflect: yes 0, borderline 38, no 0, measurements with nothing to judge 0


The hypothesis got the scores right and the reason wrong.

The judge never found the deflections relevant. Across every deflection measurement, AnswerRelevancy's judge marked 55 statements borderline, 13 no and only 2 yes. On the Faithfulness side, all 38 deflection claims came back borderline, with not a single yes. No measurement had an empty list of statements or claims either. The perfect scores came from DeepEval counting borderline as a pass, because only a no can pull a score down.

A test suite that reports skipped tests as passed behaves the same way. A deflection is a test case that skips every assertion. The run goes green, and the dashboard shows zero failures. The comparison breaks down at the report, though. A test runner at least prints how many tests it skipped. DeepEval writes its reason from the verdicts marked no, so the borderline count never shows up in it.

The shipped source makes both choices plain. The scoring helper `score_qag_verdicts` is called with `passing=(Verdict.YES, Verdict.BORDERLINE)`. The reason step keeps a verdict only `if verdict.verdict == Verdict.NO`. The Faithfulness docs can't decide which rule they follow. The calculation section counts a claim as truthful if it does not contradict the passage. The FAQ says a claim the passage does not support counts as unfaithful. The code follows the calculation section.

In [10]:
def print_calls(name):
    for kind in ["good", "deflect"]:
        call_counts = []
        for run in range(1, RUNS + 1):
            for case in CASES:
                call_counts.append(CALLS[name][run][case["id"] + "-" + kind])
        print(f"    judge calls per test case, {kind:<8} {min(call_counts)} to {max(call_counts)}")


for name in FIRST_PAIR:
    print(name)
    print_calls(name)

AnswerRelevancy
    judge calls per test case, good     3 to 3
    judge calls per test case, deflect  3 to 3
Faithfulness
    judge calls per test case, good     4 to 4
    judge calls per test case, deflect  4 to 4


AnswerRelevancy made 3 judge calls per test case, every time. One splits the reply into statements, one rules on them and one writes the reason.

Faithfulness made 4. The first two run side by side, one pulling facts from the passage and one pulling claims from the reply. The verdicts and the reason follow. That means Faithfulness re-reads the same passage from scratch for every case on every run.

The counts were identical for good answers and deflections. A deflection gets exactly as much judging as a real answer.

In [11]:
def field_reaches_judge(test_case, field, prompts):
    """True when the field's text appears word for word in at least one prompt the judge received."""
    value = getattr(test_case, field)
    if field == "retrieval_context":
        value = value[0]  # each case here carries exactly one policy passage
    for prompt in prompts:
        if value in prompt:
            return True
    return False


for name in FIRST_PAIR:
    print(name)
    for field in ["input", "actual_output", "retrieval_context"]:
        cases_seen = 0
        for case_id, test_case in TEST_CASES.items():
            if field_reaches_judge(test_case, field, PROMPTS[name][1][case_id]):
                cases_seen += 1
        print(f"    {field:<18} reaches the judge in {cases_seen} of {len(TEST_CASES)} cases")

AnswerRelevancy
    input              reaches the judge in 16 of 16 cases
    actual_output      reaches the judge in 16 of 16 cases
    retrieval_context  reaches the judge in 0 of 16 cases
Faithfulness
    input              reaches the judge in 0 of 16 cases
    actual_output      reaches the judge in 16 of 16 cases
    retrieval_context  reaches the judge in 16 of 16 cases


AnswerRelevancy sent the question and the reply to the judge in 16 of 16 cases, and the policy passage in 0 of 16. So it has no way of knowing which documents are right.

Faithfulness has the opposite gap. It sent the reply and the passage in 16 of 16 cases. It sent the question in 0 of 16, even though it declares the question as required. So it has no way of knowing which passport the traveller holds.

In [12]:
BLANKS = {"input": "", "retrieval_context": [], "actual_output": ""}
probe = TEST_CASES["c1-good"]

for name in FIRST_PAIR:
    baseline = ""
    for run in range(1, RUNS + 1):
        baseline += f"  {SCORES[name][run]['c1-good']:.2f}"
    print(f"{name} on c1-good")
    print(f"    {'nothing blanked':<26}{baseline}")
    for field, blank in BLANKS.items():
        fields = {"input": probe.input, "actual_output": probe.actual_output, "retrieval_context": probe.retrieval_context}
        fields[field] = blank
        blanked = LLMTestCase(**fields)
        scores = ""
        error_message = ""
        for run in range(1, RUNS + 1):
            try:
                result = await measure_once(name, blanked)
            except MissingTestCaseParamsError as error:
                error_message = str(error)
                break
            scores += f"  {result['score']:.2f}"
        print(f"    {field + ' blanked':<26}{scores}")
        if error_message:
            print(f"    {'':<26}  refused: {error_message}")

AnswerRelevancy on c1-good
    nothing blanked             1.00  1.00  1.00


    input blanked               0.00  0.00  0.00


    retrieval_context blanked   1.00  1.00  1.00
    actual_output blanked     
                                refused: 'actual_output' cannot be empty for the 'Answer Relevancy' metric
Faithfulness on c1-good
    nothing blanked             1.00  1.00  1.00


    input blanked               1.00  1.00  1.00


    retrieval_context blanked   1.00  1.00  1.00
    actual_output blanked     
                                refused: 'actual_output' cannot be empty for the 'Faithfulness' metric


Blanking one field at a time on c1-good confirms all of this. With the question blanked, AnswerRelevancy fell from 1.00 to 0.00 on all three runs. With the passage blanked, it held at 1.00, which is what a metric that never reads the passage should do.

Faithfulness held at 1.00 with the question blanked. More worrying, it also held at 1.00 with the passage blanked. That is a full score for an answer it had nothing to check against. The shipped verdict prompt allows a no only when the context directly contradicts a claim. An empty context contradicts nothing.

A blank reply is the only thing that stops either metric. Both raise an error before making any judge call.

In [13]:
# everything the two cards below quote, gathered in one place from the runs above
for name in FIRST_PAIR:
    report_separation(name)
    print_calls(name)

AnswerRelevancy
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 1.00
    mean score, deflections:                  0.83
    good answers passing at 0.7:            24 of 24
    deflections passing at 0.7:             16 of 24
    judge calls per test case, good     3 to 3
    judge calls per test case, deflect  3 to 3
Faithfulness
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 0.97
    mean score, deflections:                  1.00
    good answers passing at 0.7:            23 of 24
    deflections passing at 0.7:             24 of 24
    judge calls per test case, good     4 to 4
    judge calls per test case, deflect  4 to 4


### AnswerRelevancy

**The question it answers**

Does each sentence of the reply stay on the traveller's question?

**What the score means**

It is a ratio of statements marked yes or borderline to all statements. A 0.5 would mean half of a reply's sentences were ruled off the question. A perfect score is cheap, because a reply that says nothing definite gets every hedge counted as relevant. Deflections averaged 0.83, against 1.00 for good answers.

**What it cannot see**

Whether the traveller now knows what to bring. The policy passage never reaches the judge, so not a single document gets checked. A reply can hold back every document and still stay on the question.

**How it decides**

It makes 3 judge calls per test case, the same on every case we ran. The first splits the reply into statements, and the second rules on each one against the question. The third writes the reason, and it only sees the statements ruled no.

**What it needs**

The `input` and `actual_output` fields. It needs no reference answer, which makes it referenceless. A referenceless metric can score live traffic, because nobody has to write the correct answer first.

**Confused with**

ContextualRelevancy, which scores the retrieved passage against the question rather than the reply. The fields give it away: ContextualRelevancy declares `input` and `retrieval_context` and has no `actual_output` at all.

**Verdict for this use case**

Drop. It separated on 0 of 3 runs, passing 16 of 24 deflections and all 24 good answers. The deflections it did mark down were the ones with a sign-off, since every deflection holds back the documents.

### Faithfulness

**The question it answers**

Does anything in the reply contradict the policy passage?

**What the score means**

It is a ratio of claims marked yes or borderline to all claims. Borderline means the passage neither confirms nor contradicts the claim, so 1.00 means nothing was contradicted rather than everything was supported. A 0.5 would mean half the claims contradict the passage. Deflections averaged 1.00, slightly above the 0.97 for good answers.

**What it cannot see**

Which traveller is asking. The question never reaches the judge, so it cannot tell whether the reply used the right passport's rules. It cannot see a missing document either, because leaving a document out makes no claim to contradict.

**How it decides**

It makes 4 judge calls per test case, the same on every case we ran. The first two run together, one pulling facts from the passage and one pulling claims from the reply. The third rules on each claim against those facts. The fourth writes the reason from the claims ruled no.

**What it needs**

The `input`, `actual_output` and `retrieval_context` fields, although the `input` is declared and never sent to the judge. No reference answer is needed, so it can run in production wherever the retrieved passage is logged.

**Confused with**

HallucinationMetric, which checks the reply against a trusted ground-truth context. The field name gives it away: HallucinationMetric reads `context`, while Faithfulness reads `retrieval_context`.

**Verdict for this use case**

Drop as a quality gate. It separated on 0 of 3 runs and passed 24 of 24 deflections. Catching contradictions is what it was built for, and this experiment didn't test that.

## The verdict, and what has to be measured instead

We tried two candidates. The first is Faithfulness with `penalize_ambiguous_claims=True`, which counts a borderline claim as a fail. After the tallies above, it is the obvious fix to reach for.

The second is GEval, DeepEval's metric for a criterion we write ourselves. We gave it four evaluation steps, which are plain-language grading instructions the judge follows. Together they ask the visa specialist's question: does the reply name the documents this passport needs? We wrote the steps once, before any scoring, and left them alone afterwards.

In [14]:
METRIC_CLASSES["Faithfulness, strict"] = FaithfulnessMetric
EXTRA_ARGS["Faithfulness, strict"] = {"penalize_ambiguous_claims": True}

METRIC_CLASSES["GEval, documents named"] = GEval
EXTRA_ARGS["GEval, documents named"] = {
    "name": "Documents named",
    "evaluation_params": [SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT],
    # written out rather than generated, so the rubric is fixed text a visa specialist can review
    "evaluation_steps": [
        "From the retrieval context, list the documents a traveller holding the passport named in the input needs for the destination named in the input.",
        "Check which of those documents the actual output names.",
        "Lower the score for each required document the actual output leaves out, and for any document it states that the retrieval context does not require for this passport.",
        "If the actual output names none of the required documents, give the lowest score, however polite or accurate the rest of it is.",
    ],
}

SECOND_PAIR = ["Faithfulness, strict", "GEval, documents named"]
await run_suite(SECOND_PAIR)
for name in SECOND_PAIR:
    print_scores(name)
    print()

Faithfulness, strict: one score per run
    c1-good      1.00  1.00  1.00
    c1-deflect   0.00  0.00  0.00
    c2-good      0.00  0.00  0.00
    c2-deflect   0.00  0.00  0.00
    c3-good      1.00  0.67  1.00
    c3-deflect   0.00  0.00  0.00
    c4-good      1.00  1.00  1.00
    c4-deflect   0.00  0.00  0.00
    c5-good      0.50  0.50  1.00
    c5-deflect   0.00  0.00  0.00
    c6-good      0.88  0.88  0.88
    c6-deflect   0.00  0.00  0.00
    c7-good      0.50  0.50  0.40
    c7-deflect   0.00  0.00  0.00
    c8-good      1.00  1.00  1.00
    c8-deflect   0.00  0.00  0.00

GEval, documents named: one score per run
    c1-good      1.00  1.00  1.00
    c1-deflect   0.00  0.00  0.00
    c2-good      1.00  1.00  1.00
    c2-deflect   0.00  0.00  0.00
    c3-good      1.00  1.00  1.00
    c3-deflect   0.00  0.00  0.00
    c4-good      1.00  1.00  1.00
    c4-deflect   0.00  0.00  0.00
    c5-good      1.00  1.00  1.00
    c5-deflect   0.00  0.00  0.00
    c6-good      1.00  1.00  1.00

GEval scored every good answer 1.00 and every deflection 0.00, on all three runs.

The strict Faithfulness setting also scored every deflection 0.00, but it failed good answers along the way. Case c2-good scored 0.00 on all three runs, the same as its own deflection. Case c7-good landed between 0.40 and 0.50 on every run. Case c5-good scored 0.50 on two runs of three. All three replies list the documents their passage names, and GEval gave each of them 1.00.

In [15]:
for name in FIRST_PAIR + SECOND_PAIR:
    report_separation(name)
    print_calls(name)
print()
print_tallies("Faithfulness, strict")

AnswerRelevancy
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 1.00
    mean score, deflections:                  0.83
    good answers passing at 0.7:            24 of 24
    deflections passing at 0.7:             16 of 24
    judge calls per test case, good     3 to 3
    judge calls per test case, deflect  3 to 3
Faithfulness
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 0.97
    mean score, deflections:                  1.00
    good answers passing at 0.7:            23 of 24
    deflections passing at 0.7:             24 of 24
    judge calls per test case, good     4 to 4
    judge calls per test case, deflect  4 to 4
Faithfulness, strict
    runs where every deflection scored lower: 0 of 3
    mean score, good answers:                 0.74
    mean score, deflections:                  0.00
    good answers passing at 0.7:            15 of 24
    deflections passing at 0.7: 

So the strict setting separated on 0 of 3 runs. Only 15 of 24 good answers passed, and their mean fell to 0.74. Its judge marked 19 claims in good answers borderline, and the strict setting fails every one of them. We didn't print which claims those were. In short, the obvious fix swaps a blind spot on refusals for false alarms on correct answers.

GEval separated on 3 of 3 runs. It passed 24 of 24 good answers and 0 of 24 deflections, using 1 judge call per test case.

### GEval

**The question it answers**

Does the reply name the documents the policy says this traveller needs?

**What the score means**

The judge returns a raw score, which DeepEval rescales to between 0 and 1. Here it behaved as a pass or fail: good answers averaged 1.00 and deflections 0.00. Under our steps, a 0.5 would be a reply that names some of the required documents and misses others. Nothing about the score is cheap, because our last step gives a reply with no documents the lowest score.

**What it cannot see**

Whether the policy passage is right. Our first step tells the judge to take the document list from the retrieval context. It has no other source of truth.

**How it decides**

It makes 1 judge call per test case, the same on every case we ran. Normally GEval spends an extra call turning a criteria sentence into steps. We supplied the steps, so DeepEval skips that call. The judge reads the steps, the question, the reply and the passage, then returns a score and a reason.

**What it needs**

The `input`, `actual_output` and `retrieval_context` fields, because our steps name all three. It needs no reference answer, so it can score live traffic.

**Confused with**

ContextualRecall, which checks whether the retrieved passage covers a reference answer. The tell is `expected_output`: ContextualRecall needs a correct answer written by hand for every case, and it never reads the assistant's reply.

**Verdict for this use case**

Keep. It separated on 3 of 3 runs, passing 24 of 24 good answers and 0 of 24 deflections.

## What teams get wrong

- Teams read the reason text as evidence. DeepEval writes it from the no verdicts alone, which is how it came to praise deflections for documents they never named. A logged verdict list beside every score, with its borderline count, takes a day to add.
- Teams treat Faithfulness as a check that the reply is supported. It held at 1.00 with the policy passage removed. Blanking that one field on a single known case is enough to find out. A score that does not move is not checking support.
- Teams take a metric's declared fields on trust. Faithfulness declares the question and never sends it to the judge. Recording the judge's prompts for one case and searching them for each field settles this in minutes.
- Teams reach for the strict flag. Counting borderline as a fail caught every deflection and also failed correct answers. Any flag change belongs in front of known-good answers before it gates a release.
- Teams pick metrics by name instead of by the failure they need to catch. A metric earns a place in CI, the checks that run before every merge, only by separating real pairs. That means one good and one bad output for each failure the team fears most, separated on three runs.

## What the numbers license

| Metric | Separated good from deflection | Runs separated | Decision |
|---|---|---|---|
| AnswerRelevancy | No | 0 of 3 | Trust it to flag replies that drift off the question. Do not trust it to tell an answer from a polite refusal. |
| Faithfulness | No | 0 of 3 | Trust it only to say nothing contradicts the policy. A high score says nothing about whether the traveller got an answer. |
| Faithfulness, strict | No | 0 of 3 | Do not use it as a gate. It failed every refusal and also failed correct answers. |
| GEval, documents named | Yes | 3 of 3 | Trust it to say whether the reply lists the documents the policy gives. It cannot say whether the policy is right. |

A visa specialist still has to own the policy text. Every metric here grades the reply against the passage it was given. None of them would notice a rule that changed last month. They also can't tell when a transit stop needs its own visa, or when a traveller with two passports should use the other one. Those are the mistakes that get a traveller turned away at the airport. A perfect score means only that the reply matched a passage, and a person still has to sign that passage off.

So, if faithfulness and relevancy are both high, is the answer good? Here, Faithfulness passed 24 of 24 deflections and AnswerRelevancy passed 16 of 24.

A score built as the share of an answer's parts that pass can only fall when some part is judged wrong. Our deflections made claims the judge doubted every time, and Faithfulness still passed 24 of 24. A ratio metric earns its place only after it fails the most evasive output the system can produce, on every run.